# Resumen Sinteticos (Intervencional): Beta=0 vs Beta optima, agregado sobre los 11 datasets

Igual que `Resumen_Sinteticos_Observacional.ipynb`, pero sobre el **Experimento 3 intervencional**:
el modelo se entrena de forma observacional y se evalua bajo `do(x2 = 0)` (la unica intervencion de
este experimento, fija en las 11 estructuras -- a diferencia del `Y_do` de Experimento 1/2, que variaba
entre `do(Y)=0.0` y `do(Y)=1.0`, aqui no hay split por intervencion). Se compara el modelo base
(`Beta=0`) frente al modelo con la `Beta` optima, agregando sobre los 11 datasets x 10 semillas, y se
comprueba con un test de Wilcoxon pareado (por `Dataset` y `Seed`) si la mejora es significativa.

**La Beta optima no se re-selecciona aqui.** Se importa, por (Ruido, N), del resultado ya calculado
en `Resumen_Sinteticos_Observacional.ipynb` (columna `beta_optima` de
`tablas/resumen_sinteticos_observacional_beta_{gaussiano,gamma}.csv`), no de un nuevo criterio
basado en las metricas interventional (`MMD`/`RF Acc`).

Los resultados de ruido **Gaussiano** y **Gamma** se muestran en dos tablas separadas dentro de este
mismo notebook.

Todo el analisis se repite para `N=50` y `N=100`.


In [1]:
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

NOTEBOOKS_DIR = Path.cwd().parent if Path.cwd().name == "Resumenes" else Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent
TABLAS_DIR = NOTEBOOKS_DIR / "Resumenes" / "tablas"

ALPHA = 0.05
METRICAS = ["MMD", "RF Acc"]  # menor es mejor en las 2 (RF Acc bajo = generador indistinguible del real)

## Fuentes de datos (Gaussiano vs Gamma)

In [2]:
# El CSV sin sufijo es el Gaussiano; "_gamma" es el unico sufijo que existe (ver notebook observacional
# para el detalle de por que Experimento 3 no usa el sufijo "_gausian" de Experimento 1/2).
FUENTES = {
    "Gaussiano": NOTEBOOKS_DIR / "Experimento2 datasets" / "Intervencional" / "tablas" / "datasets_sinteticos_intervencional.csv",
    "Gamma": NOTEBOOKS_DIR / "Experimento2 datasets" / "Intervencional" / "tablas" / "datasets_sinteticos_intervencional_gamma.csv",
}

for nombre, path in FUENTES.items():
    assert path.exists(), f"No existe: {path}"

# Los 11 datasets del benchmark; solo se usa para medir cobertura, no para filtrar filas.
TOTAL_DATASETS_ESPERADOS = 11


## Carga de la Beta optima (importada de Resumen_Sinteticos_Observacional)

In [3]:
def cargar_beta_observacional_sinteticos():
    """Carga la Beta optima por (Ruido, N) desde los CSVs de Resumen_Sinteticos_Observacional.

    Devuelve un dict {(Ruido, N): beta_optima}.
    """
    paths = {
        "Gaussiano": TABLAS_DIR / "resumen_sinteticos_observacional_beta_gaussiano.csv",
        "Gamma": TABLAS_DIR / "resumen_sinteticos_observacional_beta_gamma.csv",
    }
    mapa = {}
    for nombre_ruido, path in paths.items():
        assert path.exists(), f"No existe: {path}"
        df_beta = pd.read_csv(path)
        for _, fila in df_beta.iterrows():
            mapa[(nombre_ruido, int(fila["N"]))] = float(fila["beta_optima"])
    return mapa


BETA_OPTIMA_OBSERVACIONAL = cargar_beta_observacional_sinteticos()

## Funcion de calculo de valores en Beta=0 y Beta=beta_optima

A diferencia de `Resumen_Sinteticos_Observacional.ipynb`, aqui la Beta no se selecciona: se recibe
ya decidida (importada de `BETA_OPTIMA_OBSERVACIONAL`). Esta funcion solo agrega, sobre TODOS los
datasets (columna `Dataset`) y semillas presentes en `df` para un N fijo, las metricas en `Beta=0`
(baseline) y en `Beta=beta_opt`.

In [4]:
def valores_para_beta(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS):
    """Calcula baseline (Beta=0) y valores en Beta=beta_opt, agregando sobre datasets x seeds.

    `beta_opt` se recibe ya decidido (importado de Resumen_Sinteticos_Observacional): esta funcion
    no selecciona ninguna Beta, solo agrega las metricas de `df` en Beta=0 y en Beta=beta_opt, para
    un N fijo.

    Devuelve (baseline: Series, valores_opt: Series).
    """
    df_n = df[df["N"] == n_filter]
    media_por_beta = df_n.groupby("Beta")[list(metrics)].mean()
    baseline = media_por_beta.loc[0.0]
    valores_opt = media_por_beta.loc[beta_opt]
    return baseline, valores_opt


## Funcion de test de Wilcoxon pareado (por Dataset y Seed)

Para cada tipo de ruido y cada N, se comparan los valores en `Beta=0` frente a `Beta=beta_optima`,
emparejados por `(Dataset, Seed)`: cada fila del CSV es un grafo (`Dataset`) entrenado con una semilla
(`Seed`) concreta, asi que el par natural es uno por cada combinacion grafo+semilla (hasta 11 x 10 =
110 pares por N) -- no solo por semilla, como en `Resumen_Observacional.ipynb` (que no tenia dimension
de grafo), ni por `(Seed, Y_do)`, como en `Resumen_Intervencional.ipynb` (que anadia una dimension de
intervencion). Test unidireccional (`alternative='less'`): H0 = no hay diferencia; H1 = el valor con
la beta optima es menor (mejor) que con beta=0.


In [5]:
def wilcoxon_beta(df: pd.DataFrame, beta_opt: float, n_filter: int, metrics=METRICAS):
    """Wilcoxon signed-rank pareado (por Dataset y Seed) entre Beta=0 y Beta=beta_opt, por metrica.

    H0: no hay diferencia. H1 (alternative='less'): el valor en beta_opt es menor (mejor) que en Beta=0.
    Devuelve (p_valores: dict metrica->p, n_pares: int).
    """
    df_n = df[df["N"] == n_filter]
    base = df_n[df_n["Beta"] == 0.0].set_index(["Dataset", "Seed"])[list(metrics)]
    opt = df_n[df_n["Beta"] == beta_opt].set_index(["Dataset", "Seed"])[list(metrics)]
    pares_comunes = sorted(set(base.index) & set(opt.index))
    base = base.loc[pares_comunes]
    opt = opt.loc[pares_comunes]

    p_valores = {}
    for m in metrics:
        try:
            _, p = wilcoxon(opt[m].values, base[m].values, alternative="less")
        except ValueError:
            p = float("nan")
        p_valores[m] = p
    return p_valores, len(pares_comunes)


## Calculo (tabla resumen + p-valores) por N, para cada tipo de ruido


In [6]:
def analizar_ruido(nombre_ruido: str, path: Path, n_filter: int, metrics=METRICAS):
    """Ejecuta valores_para_beta + wilcoxon_beta sobre TODOS los datasets presentes en `path`, a un N fijo.

    La Beta usada se toma de BETA_OPTIMA_OBSERVACIONAL[(nombre_ruido, n_filter)]; no se selecciona aqui.
    Devuelve fila_resumen: dict.
    """
    df = pd.read_csv(path)
    df_n = df[df["N"] == n_filter]
    n_datasets_presentes = df_n["Dataset"].nunique()
    cobertura = f"{n_datasets_presentes}/{TOTAL_DATASETS_ESPERADOS}"
    nota = "" if n_datasets_presentes >= TOTAL_DATASETS_ESPERADOS else f"⚠ datos parciales ({cobertura})"

    beta_opt = BETA_OPTIMA_OBSERVACIONAL[(nombre_ruido, n_filter)]
    baseline, valores_opt = valores_para_beta(df, beta_opt, n_filter=n_filter, metrics=metrics)
    p_valores, n_pares = wilcoxon_beta(df, beta_opt, n_filter=n_filter, metrics=metrics)

    fila = {
        "N": n_filter,
        "beta_optima": beta_opt,
        "MMD beta=0": baseline["MMD"],
        "MMD beta_optima": valores_opt["MMD"],
        "MMD p_valor": p_valores["MMD"],
        "RF Acc beta=0": baseline["RF Acc"],
        "RF Acc beta_optima": valores_opt["RF Acc"],
        "RF Acc p_valor": p_valores["RF Acc"],
        "n_pares": n_pares,
        "Cobertura": cobertura,
        "Criterio": f"Beta importada de Resumen_Sinteticos_Observacional (Ruido={nombre_ruido}, N={n_filter})",
        "Nota": nota,
    }
    return fila


resultados = {}
for nombre_ruido, path in FUENTES.items():
    filas = [analizar_ruido(nombre_ruido, path, n_filter) for n_filter in (50, 100)]
    resultados[nombre_ruido] = pd.DataFrame(filas)

resumen_gaussiano = resultados["Gaussiano"]
resumen_gamma = resultados["Gamma"]


## Funciones de formato y resaltado en negrita


In [7]:
COLUMNA_A_METRICA = {
    "MMD beta_optima": "MMD",
    "RF Acc beta_optima": "RF Acc",
}


def formatear(resumen: pd.DataFrame):
    cols_num = [c for c in resumen.columns if c not in ("N", "beta_optima", "Criterio", "Nota", "Cobertura")]
    fmt = resumen.copy()
    fmt[cols_num] = fmt[cols_num].round(5)
    fmt["beta_optima"] = fmt["beta_optima"].round(2)
    return fmt


def tabla_con_negrita(resumen: pd.DataFrame):
    """Tabla resumen con las celdas 'beta_optima' en negrita cuando su p-valor (Wilcoxon) < ALPHA.

    Solo aplica en la visualizacion del notebook (un CSV plano no admite negrita).
    """
    fmt = formatear(resumen)

    def resaltar(row):
        estilos = []
        for col in row.index:
            metrica = COLUMNA_A_METRICA.get(col)
            if metrica is not None and pd.notna(row[f"{metrica} p_valor"]) and row[f"{metrica} p_valor"] < ALPHA:
                estilos.append("font-weight: bold")
            else:
                estilos.append("")
        return estilos

    return fmt.style.apply(resaltar, axis=1)


## Ruido Gaussiano

In [8]:
tabla_con_negrita(resumen_gaussiano)


,N,beta_optima,MMD beta=0,MMD beta_optima,MMD p_valor,RF Acc beta=0,RF Acc beta_optima,RF Acc p_valor,n_pares,Cobertura,Criterio,Nota
0,50,0.300000,0.053110,0.051120,0.013710,0.726360,0.722950,0.157890,110,11/11,"Beta importada de Resumen_Sinteticos_Observacional (Ruido=Gaussiano, N=50)",
1,100,0.400000,0.043350,0.044090,0.351070,0.700680,0.695230,0.220330,110,11/11,"Beta importada de Resumen_Sinteticos_Observacional (Ruido=Gaussiano, N=100)",


## Ruido Gamma

In [9]:
tabla_con_negrita(resumen_gamma)


,N,beta_optima,MMD beta=0,MMD beta_optima,MMD p_valor,RF Acc beta=0,RF Acc beta_optima,RF Acc p_valor,n_pares,Cobertura,Criterio,Nota
0,50,0.300000,0.073090,0.070440,0.005600,0.761590,0.755910,0.180670,110,11/11,"Beta importada de Resumen_Sinteticos_Observacional (Ruido=Gamma, N=50)",
1,100,0.100000,0.067610,0.067420,0.049310,0.744090,0.746140,0.671950,110,11/11,"Beta importada de Resumen_Sinteticos_Observacional (Ruido=Gamma, N=100)",


## Comparacion Gaussiano vs Gamma

Beta optima elegida y p-valores por N y tipo de ruido, para ver de un vistazo si las conclusiones
cambian entre Gaussiano y Gamma.


In [10]:
comparacion = pd.concat(
    {nombre: df.set_index("N")[["beta_optima", "MMD p_valor", "RF Acc p_valor", "n_pares", "Cobertura"]]
     for nombre, df in resultados.items()},
    names=["Ruido", "N"]
)
comparacion


beta_optima  MMD p_valor  RF Acc p_valor  n_pares Cobertura
Ruido     N                                                               
Gaussiano 50           0.3     0.013714        0.157887      110     11/11
          100          0.4     0.351067        0.220335      110     11/11
Gamma     50           0.3     0.005604        0.180670      110     11/11
          100          0.1     0.049313        0.671953      110     11/11

## Guardar CSVs resumen


In [11]:
OUT_PATH_GAUSSIANO = TABLAS_DIR / "resumen_sinteticos_intervencional_beta_gaussiano.csv"
OUT_PATH_GAMMA = TABLAS_DIR / "resumen_sinteticos_intervencional_beta_gamma.csv"

resumen_gaussiano.to_csv(OUT_PATH_GAUSSIANO, index=False)
resumen_gamma.to_csv(OUT_PATH_GAMMA, index=False)

print(f"Guardado en: {OUT_PATH_GAUSSIANO}")
print(f"Guardado en: {OUT_PATH_GAMMA}")

Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_sinteticos_intervencional_beta_gaussiano.csv
Guardado en: c:\Users\aarna\Desktop\clau\TFM Claudia\kacgm-hsic\notebooks\Resumenes\tablas\resumen_sinteticos_intervencional_beta_gamma.csv
